# Project 008: Support Vector Machine (SVM) Classification Studio

**Category:** Machine Learning Basic Projects  
**Domain:** Foundational Supervised Learning & Maximum-Margin Hyperplane Geometry  
**Dataset:** Wisconsin Breast Cancer Diagnostic Dataset (569 samples, 30 features)  

---

### 📖 Theoretical Foundation

A Support Vector Machine (SVM) seeks a separating hyperplane $w \cdot x + b = 0$ that maximizes the geometric margin $\frac{2}{\|w\|_2}$ between classes while penalizing margin violations through slack variables $\xi_i \ge 0$:

$$\min_{w, b, \xi} \frac{1}{2} \|w\|^2 + C \sum_{i=1}^N \xi_i$$
$$\text{subject to } y_i (w \cdot x_i + b) \ge 1 - \xi_i, \quad \xi_i \ge 0$$

#### ⚖️ The Dual Formulation & Support Vectors:
Using Lagrange multipliers $\alpha_i$, the decision function depends solely on inner products of support vectors:

$$f(x) = \sum_{i \in \text{SV}} \alpha_i y_i K(x_i, x) + b$$

- **Linear Kernel:** $K(x, x') = x^T x'$
- **RBF (Gaussian) Kernel:** $K(x, x') = \exp(-\gamma \|x - x'\|^2)$
- **Polynomial Kernel:** $K(x, x') = (\gamma x^T x' + r)^d$

---

### 🎯 Notebook Workflow
1. **Environment Setup & Imports**: Load scikit-learn, numpy, pandas, and matplotlib.
2. **Real Dataset Loading**: Wisconsin Breast Cancer Diagnostic (569 samples, 30 features).
3. **Feature Scaling Analysis**: Demonstrate the necessity of standardization for distance-based margins.
4. **Linear SVM & Hyperplane Geometry**: Compute weight vector $\|w\|$ and exact geometric margin width $\frac{2}{\|w\|}$.
5. **Non-Linear RBF Kernel SVM**: Dual multipliers and support vector distribution.
6. **Multi-Kernel Benchmark Tournament**: Linear vs RBF vs Poly vs Sigmoid.
7. **Hyperparameter Grid Search**: Cross-validated tuning across $(C, \gamma)$ space.
8. **Champion Model Evaluation**: Confusion Matrix, ROC-AUC, Specificity, Sensitivity, and F1-Score.
9. **2D Decision Boundary & Margin Visualization**: Visualizing the decision surface and margins.

In [ ]:
# 1. Environment Setup & Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve

print("Scikit-learn and analytical libraries successfully loaded.")

In [ ]:
# 2. Real Dataset Loading: Wisconsin Breast Cancer Diagnostic
cancer_data = load_breast_cancer()
df = pd.DataFrame(cancer_data.data, columns=cancer_data.feature_names)
df['target'] = cancer_data.target
target_names = [str(n).capitalize() for n in cancer_data.target_names] # 0 = Malignant, 1 = Benign

print(f"Dataset Shape: {df.shape[0]} samples x {len(cancer_data.feature_names)} features")
print(f"Target Classes: {target_names}")
print(df['target'].value_counts().rename(index={0: 'Malignant (0)', 1: 'Benign (1)'}))
df.head()

In [ ]:
# 3. Feature Scaling & Train/Test Stratified Split
X = df[cancer_data.feature_names].values
y = df['target'].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"Training Partition: {X_train.shape[0]} samples")
print(f"Holdout Test Partition: {X_test.shape[0]} samples")

In [ ]:
# 4. Linear Support Vector Machine & Geometric Margin
linear_svm = SVC(kernel='linear', C=1.0, random_state=42)
linear_svm.fit(X_train, y_train)

w = linear_svm.coef_[0]
w_norm = np.linalg.norm(w)
margin_width = 2.0 / w_norm
n_sv_linear = len(linear_svm.support_)

print("=== Linear SVM Hyperplane Analysis ===")
print(f"Number of Support Vectors: {n_sv_linear} ({(n_sv_linear/len(X_train))*100:.1f}% of train set)")
print(f"Hyperplane Norm ||w||: {w_norm:.4f}")
print(f"Exact Geometric Margin Width (2 / ||w||): {margin_width:.4f}")
print(f"Test Accuracy: {linear_svm.score(X_test, y_test):.4f}")

In [ ]:
# 5. Non-Linear Radial Basis Function (RBF / Gaussian) SVM
rbf_svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
rbf_svm.fit(X_train, y_train)

n_sv_rbf = len(rbf_svm.support_)
print("=== RBF Gaussian Kernel SVM ===")
print(f"Total Support Vectors: {n_sv_rbf} ({(n_sv_rbf/len(X_train))*100:.1f}% of train set)")
print(f"Support Vectors per Class [Malignant, Benign]: {rbf_svm.n_support_.tolist()}")
print(f"Test Accuracy: {rbf_svm.score(X_test, y_test):.4f}")

In [ ]:
# 6. Multi-Kernel Benchmark Tournament
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
results = []

for k in kernels:
    model = SVC(kernel=k, C=1.0, probability=True, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    
    results.append({
        'Kernel': k.upper(),
        'Accuracy': accuracy_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, proba),
        'Support Vectors': len(model.support_),
        'SV Ratio (%)': (len(model.support_) / len(X_train)) * 100.0
    })

tournament_df = pd.DataFrame(results)
tournament_df

In [ ]:
# 7. Hyperparameter Grid Search (C vs Gamma)
param_grid = {
    'C': [0.1, 1.0, 10.0, 100.0],
    'gamma': ['scale', 0.001, 0.01, 0.1, 1.0]
}

grid = GridSearchCV(SVC(kernel='rbf', probability=True, random_state=42), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print(f"Optimal Hyperparameters: {grid.best_params_}")
print(f"Best 5-Fold Cross-Validation Accuracy: {grid.best_score_:.4f}")

In [ ]:
# 8. Champion Model Evaluation (Confusion Matrix & Classification Report)
champion_model = grid.best_estimator_
y_pred = champion_model.predict(X_test)
y_proba = champion_model.predict_proba(X_test)[:, 1]

print("=== Classification Report on Unseen Holdout ===")
print(classification_report(y_test, y_pred, target_names=target_names))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black', fontweight='bold')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels([f'Pred {n}' for n in target_names])
ax.set_yticklabels([f'True {n}' for n in target_names])
ax.set_title("Confusion Matrix")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# 9. 2D Decision Space & Margin Visualization on First Two Principal Features
X_2d = X_train[:, :2]
model_2d = SVC(kernel='rbf', C=1.0, gamma='scale').fit(X_2d, y_train)

x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

Z = model_2d.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, levels=20, cmap='coolwarm', alpha=0.3)
plt.contour(xx, yy, Z, levels=[-1.0, 0.0, 1.0], colors=['crimson', 'black', 'green'], linestyles=['--', '-', '--'], linewidths=[1.5, 2.5, 1.5])
plt.scatter(X_2d[y_train == 0, 0], X_2d[y_train == 0, 1], color='red', label=target_names[0], edgecolors='k', alpha=0.7)
plt.scatter(X_2d[y_train == 1, 0], X_2d[y_train == 1, 1], color='blue', label=target_names[1], edgecolors='k', alpha=0.7)
plt.scatter(X_2d[model_2d.support_, 0], X_2d[model_2d.support_, 1], s=120, facecolors='none', edgecolors='gold', linewidths=2, label='Support Vectors')
plt.title('SVM Decision Boundary with Margin Gutters and Support Vectors')
plt.xlabel('Mean Radius (Standardized)')
plt.ylabel('Mean Texture (Standardized)')
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()